In [6]:
import pandas as pd
import numpy as np
from pybaseball import batting_stats, fielding_stats
from sklearn.ensemble import RandomForestRegressor
import os

# 1. Directory Setup
base_dir = os.path.dirname(os.getcwd())
input_path = os.path.join(base_dir, 'data', 'processed', 'hitter_classified_data.csv')
output_final_path = os.path.join(base_dir, 'data', 'processed', 'hitter_final_results.csv')

# 2. Load Classified Data
if not os.path.exists(input_path):
    print(f"❌ Input file not found: {input_path}")
else:
    df = pd.read_csv(input_path)

    # 3. Data Augmentation (Fetch supplemental stats for weighting)
    print("🌐 Fetching supplemental stats for ML weighting (SB, OAA, etc.)...")
    try:
        # Use qual=150 to ensure we don't lose any of our 290 players
        b_stats = batting_stats(2024, qual=150)[['Name', 'PA', 'SB', 'xwOBA']].copy()
        b_stats.columns = ['full_name', 'pa', 'sb', 'xwoba_target']
        
        f_stats = fielding_stats(2024)[['Name', 'DRS', 'Pos']].copy()
        f_stats.columns = ['full_name', 'oaa', 'primary_pos']
        f_stats = f_stats.drop_duplicates(subset=['full_name'], keep='first')

        # Clean existing columns to prevent merge conflicts
        cols_to_drop = ['pa', 'sb', 'oaa', 'primary_pos', 'xwoba_target']
        df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

        # Merge with official stats
        df = pd.merge(df, b_stats, on='full_name', how='left')
        df = pd.merge(df, f_stats, on='full_name', how='left')

        # Handle missing values for merged columns
        df['pa'] = pd.to_numeric(df['pa'], errors='coerce').fillna(0).astype(int)
        df['sb'] = df['sb'].fillna(0)
        df['oaa'] = df['oaa'].fillna(0)
        df['primary_pos'] = df['primary_pos'].fillna('DH')

        # Final Sample Filter
        df = df[df['pa'] >= 200].copy()
        print(f"✅ Data Integration Complete (Sample Size: {len(df)})")
        
    except Exception as e:
        print(f"⚠️ Error during data augmentation: {e}")

    # 4. Feature Engineering: Utility Score
    # Assigning 1-4 points based on defensive versatility
    def get_utility_score(pos):
        pos_str = str(pos).upper()
        if '/' in pos_str or '-' in pos_str: return 4
        if '2B' in pos_str or 'SS' in pos_str: return 3
        if 'CF' in pos_str or 'C' in pos_str: return 2
        return 1

    df['utility'] = df['primary_pos'].apply(get_utility_score)
    
    # Standardization helper function
    def std_scale(series):
        if series.std() == 0 or len(series) <= 1: return series * 0
        return (series - series.mean()) / (series.std() + 1e-9)

    # 5. 🤖 ML-Based Weight Extraction & Rank Score Calculation
    def calculate_rf_weights(target_df, features, target_col='xwoba_target'):
        """Extracts feature importance using Random Forest as an objective weighting mechanism."""
        X = target_df[features].fillna(0)
        y = target_df[target_col].fillna(0)
        rf = RandomForestRegressor(n_estimators=100, random_state=42)
        rf.fit(X, y)
        return dict(zip(features, rf.feature_importances_))

    target_types = ['Type A', 'Type B', 'Type C', 'Type D']
    df['rank_score'] = 0.0

    print("\n🔍 AI-Derived Weights by Hitter Type:")
    
    for t in target_types:
        mask = df['hitter_type'] == t
        if not mask.any(): continue
        
        if t == 'Type A':
            features = ['avg', 'k_pct', 'chase_pct', 'slg']
            weights = calculate_rf_weights(df[mask], features)
            # Apply negative weight to 'K%' and 'Chase%' as they are inverse productivity metrics
            df.loc[mask, 'rank_score'] = (std_scale(df.loc[mask, 'avg']) * weights['avg'] - 
                                          std_scale(df.loc[mask, 'k_pct']) * weights['k_pct'] - 
                                          std_scale(df.loc[mask, 'chase_pct']) * weights['chase_pct'] +
                                          std_scale(df.loc[mask, 'slg']) * weights['slg'])
            
        elif t == 'Type B':
            features = ['avg_ev', 'slg', 'sweet_spot_pct']
            weights = calculate_rf_weights(df[mask], features)
            df.loc[mask, 'rank_score'] = sum(std_scale(df.loc[mask, f]) * weights[f] for f in features)
            
        elif t == 'Type C':
            features = ['xwoba', 'ops', 'xbh']
            weights = calculate_rf_weights(df[mask], features)
            df.loc[mask, 'rank_score'] = sum(std_scale(df.loc[mask, f]) * weights[f] for f in features)
            
        elif t == 'Type D':
            # Type D focuses on defensive/baserunning contribution (Targeting OAA)
            features = ['sb', 'utility']
            weights = calculate_rf_weights(df[mask], features, target_col='oaa')
            df.loc[mask, 'rank_score'] = (std_scale(df.loc[mask, 'sb']) * weights['sb'] + 
                                          std_scale(df.loc[mask, 'oaa']) * 1.5 + # Defensive anchor weight
                                          std_scale(df.loc[mask, 'utility']) * weights['utility'])
        
        # Display derived weights for transparency
        formatted_weights = {k: round(v, 3) for k, v in weights.items()}
        print(f" > {t} weights: {formatted_weights}")

    # 6. Rank Generation & Display
    print("\n" + "="*50)
    print("🏆 ML-Weighted Hitter Ranking (Min 200 PA)")
    print("="*50)
    
    for t in target_types:
        print(f"\n📍 {t} Leaders:")
        subset = df[df['hitter_type'] == t].sort_values('rank_score', ascending=False)
        if not subset.empty:
            display_cols = ['full_name', 'pa', 'rank_score']
            if t == 'Type D': 
                display_cols.insert(2, 'sb')
                display_cols.insert(3, 'oaa')
            print(subset[display_cols].head(5).to_string(index=False))

    # 7. Export Final Results
    df.to_csv(output_final_path, index=False)
    print(f"\n🏁 Final analysis exported to: {output_final_path}")

🌐 Fetching supplemental stats for ML weighting (SB, OAA, etc.)...
✅ Data Integration Complete (Sample Size: 290)

🔍 AI-Derived Weights by Hitter Type:
 > Type A weights: {'avg': 0.092, 'k_pct': 0.132, 'chase_pct': 0.185, 'slg': 0.591}
 > Type B weights: {'avg_ev': 0.16, 'slg': 0.784, 'sweet_spot_pct': 0.056}
 > Type C weights: {'xwoba': 0.85, 'ops': 0.101, 'xbh': 0.049}
 > Type D weights: {'sb': 0.735, 'utility': 0.265}

🏆 ML-Weighted Hitter Ranking (Min 200 PA)

📍 Type A Leaders:
      full_name  pa  rank_score
    Kyle Tucker 339    2.923574
   Mookie Betts 516    1.918620
Freddie Freeman 638    1.384686
    Steven Kwan 540    1.305659
    Jonah Bride 272    1.159849

📍 Type B Leaders:
      full_name  pa  rank_score
    Aaron Judge 704    3.419817
  Shohei Ohtani 731    2.723127
Kerry Carpenter 296    1.939477
   Brent Rooker 614    1.506463
  Marcell Ozuna 688    1.499882

📍 Type C Leaders:
       full_name  pa  rank_score
       Juan Soto 713    3.706036
     Ketel Marte 583    1.